# Base final e KPIs do Frigus

## Objetivo
Preparar uma base analítica de transações para o dashboard

## Entrada
workspace.frigus_cleaned.transactions

## Saídas previstas
- workspace.frigus_bi.transacoes_dashboard
- workspace.frigus_bi.kpis_transacoes_diarios

## Granularidade
A base do dashboard terá uma linha por transação

## Filtros previstos
- Período: data de criação da transação
- Método de pagamento

## Indicadores
- Quantidade de transações
- Valor total das transações
- Valor médio por transação
- Evolução diária
- Distribuição por pagamento e status

In [0]:
origem = spark.table("workspace.frigus_cleaned.transactions")
total_origem = origem.count()

assert total_origem > 0, (
    "A tabela de transacoes esta vazia"
)

invalidos = spark.sql("""
    SELECT COUNT(*) AS total
    FROM workspace.frigus_cleaned.transactions
    WHERE id IS NULL
       OR created_at IS NULL
       OR amount IS NULL
""").first()["total"]

assert invalidos == 0, (
    f"Existem {invalidos} registros sem id, data ou valor. "
    "Revise esses registros antes de continuar"
)

duplicados = spark.sql("""
    SELECT id
    FROM workspace.frigus_cleaned.transactions
    GROUP BY id
    HAVING COUNT(*) > 1
""").count()

assert duplicados == 0, (
    "Existem IDs de transacao duplicados"
)

print(
    f"Entrada validada: {total_origem} transacoes"
)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.frigus_bi")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.frigus_bi.transacoes_dashboard
    USING DELTA
    AS
    SELECT
        id AS transacao_id,

        CAST(created_at AS DATE) AS data_transacao,

        COALESCE(
            NULLIF(TRIM(payment_method), ''),
            'Nao informado'
        ) AS metodo_pagamento,

        COALESCE(
            NULLIF(TRIM(status), ''),
            'Nao informado'
        ) AS status_transacao,

        amount AS valor_transacao,

        attempts AS tentativas,

        CASE
            WHEN attempts IS NULL THEN NULL
            WHEN attempts > 1 THEN 1
            ELSE 0
        END AS teve_retry,

        CASE
            WHEN amount < 0 THEN '0. Negativo'
            WHEN amount < 20 THEN '1. De 0 a menos de 20'
            WHEN amount < 50 THEN '2. De 20 a menos de 50'
            WHEN amount < 100 THEN '3. De 50 a menos de 100'
            ELSE '4. A partir de 100'
        END AS faixa_valor,

        _ingested_at AS ingerido_em,

        CURRENT_TIMESTAMP() AS atualizado_em

    FROM workspace.frigus_cleaned.transactions
""")

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE workspace.frigus_bi.kpis_transacoes_diarios
    USING DELTA
    AS
    SELECT
        data_transacao,
        metodo_pagamento,
        status_transacao,

        COUNT(*) AS quantidade_transacoes,

        SUM(valor_transacao) AS valor_total,

        AVG(valor_transacao) AS valor_medio,

        SUM(teve_retry) AS transacoes_com_retry,

        COUNT(teve_retry)
            AS transacoes_com_tentativas_informadas

    FROM workspace.frigus_bi.transacoes_dashboard

    GROUP BY
        data_transacao,
        metodo_pagamento,
        status_transacao
""")

In [0]:
total_final = spark.table(
    "workspace.frigus_bi.transacoes_dashboard"
).count()

assert total_final == total_origem, (
    f"Contagem divergente: origem={total_origem}, "
    f"base final={total_final}."
)

display(spark.sql("""
    SELECT
        COUNT(*) AS quantidade_transacoes,
        MIN(data_transacao) AS primeira_data,
        MAX(data_transacao) AS ultima_data,
        SUM(valor_transacao) AS valor_total,
        AVG(valor_transacao) AS valor_medio,
        MAX(atualizado_em) AS atualizado_em
    FROM workspace.frigus_bi.transacoes_dashboard
"""))

display(spark.sql("""
    SELECT *
    FROM workspace.frigus_bi.kpis_transacoes_diarios
    ORDER BY data_transacao DESC
    LIMIT 30
"""))

print("base final e KPIs foram gerados")